# WarheadVerify: arms-control claim verification (corpus, baselines, cross-encoder, LLM agent)

Companion notebook for [MHKhan50/warheadverify-agentbench](https://github.com/MHKhan50/warheadverify-agentbench).
Claims about nuclear arms-control treaty status (PTBT, NPT, CTBT, TPNW) are verified against yearly status records from
the UNODA Treaties Database (published by Our World in Data, CC BY 4.0). Public treaty status only; no weapons information.

Kaggle settings: **GPU T4**, **Internet on**. *Save Version → Save & Run All*. A smoke test runs first; the full run takes
about 1 hour. Outputs: `/kaggle/working/reports/` and `/kaggle/working/warheadverify_reports.zip`.

In [1]:
import os, subprocess, sys
REPO_URL = "https://github.com/MHKhan50/warheadverify-agentbench.git"
os.chdir("/kaggle/working")
if not os.path.exists("warheadverify-agentbench"):
    subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
os.chdir("warheadverify-agentbench")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[gpu]"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
RAW, DATA, OUT = "/kaggle/working/data/raw", "/kaggle/working/data/processed", "/kaggle/working/reports"
def wv(*args):
    """Run one CLI step, stream its output and stop with the error text if it fails."""
    proc = subprocess.Popen([sys.executable, "-m", "warheadverify.cli", *args, "--raw-dir", RAW, "--data-dir", DATA, "--out", OUT],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in proc.stdout:
        print(line, end="", flush=True); lines.append(line)
    if proc.wait():
        raise RuntimeError("step failed: " + " ".join(args) + "\n" + "".join(lines[-30:]))

86c1d6e Cross-encoder: load weights in fp32 before mixed-precision training



## 1. Download the status tables (manifest with SHA-256) and build the corpus

In [2]:
wv("download")
wv("build")

{
  "source": "UNODA Treaties Database via Our World in Data (CC BY 4.0)",
  "files": {
    "PTBT": {
      "url": "https://ourworldindata.org/grapher/partial-nuclear-test-ban-treaty.csv?v=1&csvType=full&useColumnShortNames=false",
      "sha256": "dbccf468914dacbf5cc483985efd7763bad976096b215399cc79c204bdb8384b",
      "rows": 12411,
      "retrieved_utc": "2026-11-20T20:00:02+00:00"
    },
    "NPT": {
      "url": "https://ourworldindata.org/grapher/nuclear-non-proliferation-treaty.csv?v=1&csvType=full&useColumnShortNames=false",
      "sha256": "3cc33beffc6a6148eb59407251e7e699ba4f24b0a45d821e35c31b46358ee188",
      "rows": 11426,
      "retrieved_utc": "2026-11-20T20:00:03+00:00"
    },
    "CTBT": {
      "url": "https://ourworldindata.org/grapher/comprehensive-nuclear-test-ban-treaty.csv?v=1&csvType=full&useColumnShortNames=false",
      "sha256": "a3f8e3b910db61600fe3f2980b6f7f6cffeb19b4a241a89df646b7855924aa68",
      "rows": 5910,
      "retrieved_utc": "2026-11-20T20:00:03+

## 2. CPU: retrieval, lexical baselines, lookup agent

In [3]:
wv("cpu")

/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (60) reached and the optimization hasn't converged yet.
  warnings.warn(
{
  "retrieval_tfidf": {
    "recall@1": 0.8384094754653131,
    "recall@3": 0.9767343485617598,
    "recall@5": 0.9978849407783418,
    "mrr": 0.9063522278623802,
    "n": 4728
  },
  "country": {
    "majority": 0.18074899411946765,
    "claim_lr": 0.732525051209655,
    "pair_lr": 0.7245897921298483,
    "pair_mlp": 0.828448258513886,
    "pair_lr_gold": 0.7219407013556296,
    "lookup_agent": 1.0,
    "by_kind_lookup_acc": null
  },
  "treaty": {
    "majority": 0.1660313426514189,
    "claim_lr": 0.583395162415045,
    "pair_lr": 0.5870025457669161,
    "pair_mlp": 0.623380826940846,
    "pair_lr_gold": 0.5699818153419075,
    "lookup_agent": 1.0,
    "by_kind_lookup_acc": null
  }
}


## 3. Smoke test of the GPU step (1 epoch, 1 seed, 30 LLM claims), then the full run

In [4]:
wv("gpu", "--seeds", "0", "--epochs", "1", "--llm-claims", "30")
wv("gpu")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2726.97it/s]

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 10388.75it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
pooler.dense.weight                 

## 4. Summary and zip

In [5]:
wv("summary")
import shutil, glob
shutil.copytree(DATA, OUT + "/corpus", dirs_exist_ok=True)
shutil.copy(RAW + "/manifest.json", OUT + "/corpus/manifest.json")
shutil.make_archive("/kaggle/working/warheadverify_reports", "zip", OUT)

| Verifier | Country-held-out test | Treaty-held-out test (TPNW) |
|---|---|---|
| Majority class | 0.181 [0.169, 0.191] | 0.166 [0.156, 0.176] |
| Claim only, TF-IDF LR | 0.733 [0.700, 0.766] | 0.583 [0.558, 0.610] |
| Claim + retrieved record, TF-IDF LR | 0.725 [0.691, 0.759] | 0.587 [0.566, 0.611] |
| Claim + retrieved record, TF-IDF MLP | 0.828 [0.788, 0.867] | 0.623 [0.596, 0.651] |
| Claim + gold record, TF-IDF LR | 0.722 [0.690, 0.756] | 0.570 [0.543, 0.599] |
| Cross-encoder (DeBERTa-v3-small), 3 seeds, mean ± SD | 0.910 ± 0.004 | 0.836 ± 0.139 |
| Zero-shot NLI (DeBERTa-v3-small) | 0.413 [0.372, 0.451] | 0.466 [0.440, 0.491] |
| Lookup agent (parser + record check) | 1.000 [1.000, 1.000] | 1.000 [1.000, 1.000] |

Retrieval (TF-IDF): recall@1 0.838, recall@3 0.977, recall@5 0.998, mrr 0.906
Retrieval (e5-small): recall@1 0.944, recall@3 0.997, recall@5 0.999, mrr 0.970

Agent pipeline (Qwen/Qwen2.5-1.5B-Instruct, 600 country-held-out test claims):
- LLM verifier raw: 0.400 [0.3

'/kaggle/working/warheadverify_reports.zip'